# Pesquisa no corpus e QA de resposta curta

Partimos do tópico, investigamos recortes, construímos um documento amplo auditado e extraímos unidades com alvos curtos. Só depois formulamos perguntas fechadas.

O notebook usa o novo motor `qa_research`. Não envia datasets externos ou questões de exemplo. A configuração inicial é um **piloto de dois recortes**, sem pretensão de cobrir todo o tópico.


In [ ]:
from pathlib import Path
import json
import sys

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
ROOT = next((p for p in candidates if (p / 'pipeline/qa_research/config.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Abra este notebook dentro do repositório MCQ_CONTEXT_GEN.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import Markdown, display
from pipeline.qa_research.run import load_config, inspect, execute
from pipeline.inventario.textutil import read_json, read_jsonl


## 1. Tópico e limites

O arquivo contém o escopo completo de Produtos químicos. Para uma investigação maior, troque pelo perfil `produtos_quimicos.json` e escolha uma nova pasta de saída.

Sem manifesto, a busca utiliza apenas trechos originais preservados nas seleções existentes: a cobertura é parcial. Se tiver o manifesto do corpus completo instalado, informe seu caminho antes de executar a pesquisa.


In [ ]:
CONFIG_PATH = ROOT / 'pipeline/qa_research/configs/piloto_produtos_quimicos.json'
CONFIG = load_config(CONFIG_PATH)
# Opcional: CONFIG.manifest = str(ROOT / 'caminho/para/manifest.json')
# Opcional: CONFIG.output = str(ROOT / 'pipeline/qa_research/saida_nova_investigacao')
CONFIG.validate()
OUT = Path(CONFIG.output)
print(CONFIG.topic['titulo'])
print(CONFIG.topic['escopo'])
print(f'Recortes: {CONFIG.initial_facets} iniciais, até {CONFIG.max_facets} no total')
print(f'Meta QA: {CONFIG.target_qa}; resposta: até {CONFIG.answer_words} palavras')
print(f'Orçamento cumulativo: {CONFIG.max_calls} tentativas de API')


## 2. Inspeção local

Esta célula não chama modelos, não lê credenciais e não cria o índice. Os recortes concretos serão propostos pelo planejador na pesquisa.


In [ ]:
preview = inspect(CONFIG)
display({'origens': preview['origens'], 'manifest': preview['manifest']})


## 3. Pesquisa e documento auditado

**Esta célula faz chamadas aos deployments configurados.** Planejador e pesquisadores organizam os recortes; redator escreve capítulos; auditor verifica os originais. Novas buscas e uma revisão são limitadas pela configuração. A mesma pasta permite retomar uma execução interrompida sem repetir chamadas concluídas.


In [ ]:
resumo_pesquisa = execute(CONFIG, stage='pesquisa')
display(resumo_pesquisa)


## 4. Examinar recortes, cobertura e documento

Blocos que falharam na auditoria não entram no documento publicado. Lacunas e recortes não investigados continuam registrados. A aprovação é do modelo; a revisão humana permanece pendente.


In [ ]:
plano = read_json(OUT / 'plano.json') or {}
for recorte in plano.get('recortes', []):
    print(recorte['id'], '|', recorte['titulo'])
    print('  ', recorte['servico'], '×', recorte['fenomeno_produto'], '×', recorte['decisao'])
    print('  origem:', recorte['origem'])
display(read_json(OUT / 'cobertura.json'))
doc_path = OUT / 'documento_auditado.md'
if doc_path.exists():
    display(Markdown(doc_path.read_text(encoding='utf-8')))


## 5. Extrair conhecimento e produzir QA

**Esta célula também faz chamadas aos modelos.** O extrator decompõe os blocos admitidos em unidades e escolhe alvos de resposta curta. Auditoria filtra unidades ambíguas, subjetivas ou sem contexto. O autor formula a pergunta; outra chamada tenta respondê-la sem receber o gabarito; auditor adversarial procura respostas diferentes igualmente válidas.

Não há obrigação de atingir a meta se faltarem unidades aprováveis. Sinônimos são variantes da mesma resposta, não ambiguidade por si só.


In [ ]:
resumo_qa = execute(CONFIG, stage='qa')
display(resumo_qa)


## 6. Ler os pares aprovados e rastrear um item

`qa.csv` contém somente pergunta e resposta. O JSONL guarda evidências e pareceres. Nenhuma célula injeta esses pares como exemplos em gerações posteriores.


In [ ]:
aprovadas = read_jsonl(OUT / 'qa_aprovadas.jsonl')
for i, item in enumerate(aprovadas, 1):
    print(f'{i}. {item["pergunta"]}')
    print('Resposta:', item['resposta'])
    print()
if not aprovadas:
    print('Nenhuma QA aprovada nesta execução. Consulte relatorio.md e qa_nao_aprovadas.jsonl.')


In [ ]:
if aprovadas:
    item = aprovadas[0]
    fontes = {e['id']: e for e in read_jsonl(OUT / 'evidencias.jsonl')}
    print('Proposição:', item['unidade']['proposicao'])
    print('Condições:', item['unidade']['condicoes'])
    for suporte in item['unidade']['suporte']:
        fonte = fontes[suporte['evidencia_id']]
        print('Fonte:', fonte.get('title'), '|', fonte['chunk_id'])
        print('Citação original:', suporte['citacao'])
    print('Resolução independente:', item['resolucao_independente']['resposta'])
    print('Auditoria:', item['auditoria']['parecer'])


## Materiais de revisão

- `documento_auditado.md`: documento amplo admitido.
- `unidades.jsonl`: conhecimento extraído e alvos de resposta.
- `qa.csv`: pares aprovados para uso.
- `qa_nao_aprovadas.jsonl`: rejeições e pendências.
- `cobertura.json`: alcance, lacunas e descobertas ainda não investigadas.
- `chamadas/`: entradas, prompts e saídas para auditoria.

O plano metodológico fica em `docs/plano_qa_research.md`. Resposta determinada no contexto e aprovação automatizada não constituem garantia universal de unicidade ou correção científica.
